In [1]:
import pandas as pd

customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

# Usunięcie zduplikowanych kodów pocztowych z bazy geolokacyjnej
# (czasem jeden kod pocztowy ma przypisane kilka współrzędnych np. środek i obrzeża strefy)
geo_unique = geolocation.groupby('geolocation_zip_code_prefix').first().reset_index()

# Połączenie klientów z ich koordynatami
merged_data = pd.merge(
    customers,
    geo_unique[['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']],
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

# Na koniec warto ograniczyć zbiór do jednego, gęstego miasta (np. Sao Paulo - "SP"),
# żeby algorytm DBSCAN miał szansę zadziałać na realistycznym wycinku
sao_paulo_data = merged_data[merged_data['customer_state'] == 'SP']

In [2]:
sao_paulo_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409.0,-20.509897,-47.397866
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,9790.0,-23.726853,-46.545746
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1151.0,-23.527788,-46.660310
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,8775.0,-23.496930,-46.185352
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,13056.0,-22.987222,-47.151073


In [3]:
sao_paulo_data.to_csv('../data/processed/sao_paulo_deliveries.csv', index=False)